 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison
Name           : Selvi Ganesh Kumar                <br>
Student Number : L00196360          <br>
Due Date       : 12th May 2026                <br>
Assignment     :             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [5]:
# pip installs

requirements = """
opencv-python
pytesseract
numpy
matplotlib
pandas
pymupdf
easyocr
tqdm
pillow
nltk
spacy
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

In [6]:
#pip install -r requirements.txt

## Imports

In [8]:
# imports

import os
import zipfile

import numpy as np
import pandas as pd

import cv2
import matplotlib.pyplot as plt

import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

import pymupdf
import fitz

from tqdm import tqdm

# Text Preprocessing:

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re

from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

import spacy
#!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\selvi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\selvi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\selvi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Support Functions

In [10]:
# code here

def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.threshold(blur, 150, 255, cv2.THRESH_BINARY)[1]
    return thresh

# NLP

## Sub Heading 1

## code

In [13]:
# The dataset has different filtypes: .jpg (images), .bmp (images) and .pdf (documents)
# ZIP → extract → detect file type → extract text → NLP → output

### Image loading

In [15]:
# STEP 1: Extract ZIP
import zipfile
import os

path = r"C:\Education\M.Sc Documents\Computer Vision & NLP\CA2\documents_ca2.zip"

with zipfile.ZipFile(path, "r") as zip_ref:
    zip_ref.extractall("docs")

In [16]:
# To view the extracted files from the zip.

folder = "docs"

for root, dirs, files in os.walk(folder):
    print("Folder:", root)
    print("Files:", files)


Folder: docs
Files: []
Folder: docs\documents_ca2
Files: ['1921_doc_1.pdf', '1921_doc_sign.pdf', '1921_doc_sig_image.png', 'aloevera_back.jpg', 'cartoon_drawing.pdf', 'drawing_using_words.bmp', 'forms.jpg', 'know_receipe.pdf', 'research_paper.pdf', 'scikit_learn.pdf', 'trinity_brochure.pdf']


In [17]:
# step 2: Separate file types and storing them in their own variety, for further processing

jpg_files = []
bmp_files = []
pdf_files = []

for root, dirs, files in os.walk(folder):
    for file in files:
        path = os.path.join(root, file)

        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            jpg_files.append(path)

        elif file.lower().endswith(".bmp"):
            bmp_files.append(path)

        elif file.lower().endswith(".pdf"):
            pdf_files.append(path)

print("JPG:", len(jpg_files))
print("BMP:", len(bmp_files))
print("PDF:", len(pdf_files))

JPG: 3
BMP: 1
PDF: 7


### OCR using Tesseract on images

In [19]:
# step 3: storing the images in the images files to process images (ocr)

image_files = jpg_files + bmp_files

results = []

for file in image_files:
    img = cv2.imread(file)

    processed = preprocess_image(img)
    custom_config = r'--oem 1 --psm 6'

    text = pytesseract.image_to_string(processed, config = custom_config)

    results.append({
        "file": file,
        "type": "image",
        "text": text
    })



In this project, a hybrid approach was adopted for processing PDF documents to ensure robust and accurate text extraction across different document types. PDFs can broadly be categorised into two types: text-based PDFs, which contain embedded digital text, and scanned PDFs, which are essentially images of documents and do not contain extractable text. To handle both cases effectively, the system first attempts to extract text directly using PyMuPDF, which is efficient and preserves formatting when digital text is available. If no text is detected on a page (i.e., the extracted content is empty or minimal), the system assumes the page is image-based and applies Optical Character Recognition (OCR) using Tesseract. In this case, the PDF page is converted into an image and processed to extract text. This hybrid strategy improves both performance and accuracy by avoiding unnecessary OCR on text-based documents while still enabling text extraction from scanned or image-based PDFs. It also demonstrates a practical integration of document analysis techniques, aligning with real-world multi-modal document understanding systems.

In [21]:
# handling the pdf files separately, either by converting into image for scanned pdf, or by extracting the text directly,
# incase of read text based pdf, them the text is extracted:


for file in pdf_files:
    doc = fitz.open(file)

    text = ""  # one full document text

    for page in doc:
        extracted = page.get_text()

        if extracted.strip():
            text += extracted + "\n"

        else:
            pix = page.get_pixmap(dpi=200)

            img = np.frombuffer(pix.samples, dtype=np.uint8)
            img = img.reshape(pix.height, pix.width, pix.n)

            custom_config = r'--oem 1 --psm 6'
            text += pytesseract.image_to_string(img, config=custom_config) + "\n"

    results.append({
        "file": file,
        "type": "pdf",
        "text": text
    })

### Basic text extraction from visual data

In [23]:
results


[{'file': 'docs\\documents_ca2\\1921_doc_sig_image.png',
  'type': 'image',
  'text': "OPiS an ndnatoe Several Seorctanar |\nVeh Jacuary 1921.\nThe Senretary for Piranate 2 Vv\na Tiere,\n\nWith reference to your note of the 3th ultima J fare rade\nservers) afforta to obte:n ‘rom Messrs O'Marony and Stanahan « reSurn\nof thelr expersea in nenmmestéon sith the frera: of the late Lord\nMajor of Dirk. I Pave at Taet heard from fooes O'Com.laceh or totale\nof Mr Starehan that ie doe@ not prepose to pat in any clatm of\ncarrot ret ery Sire: reply frow Br O'Mahony but 1 tal a note from\nMre O’Matony trie ceming in stich abe paye that re has written ter to\ntell me to [sdae ria expenses by the others sa:t :a- A Tetum ticket\nto Gork conte @2.8.¢ and as be hed to stay thrae daze in Cork 7\npresume £4 vonls tes fair eatizate.\n\nT send you ‘erewith the cliaize ekish tare toe. rece: ved an |\nag afracd there la +o Scope of getting anzthics wore definite at presest\nand .t le teara]7 fate to tnep 

This pipeline processes a mixed dataset of images and PDF files to extract text for further NLP use. PDF files are handled using PyMuPDF, which directly extracts embedded digital text, resulting in highly accurate outputs. Image files are processed using Tesseract OCR, which converts visual content into machine-readable text. The quality of extracted text from images varies depending on the type of image: clear printed images (such as charts, diagrams, or scanned pages) produce better results, while handwritten images, blurred photos, or noisy scans often lead to incomplete or incorrect recognition due to OCR limitations. Therefore, differences in output quality are not caused by errors in the code, but by the nature and quality of the input data and the constraints of OCR technology when working with complex visual content.

Most of the images include scanned documents, photos (not scans), handwriting, artistic layouts, low contrast backgrounds. Without strong preprocessing, Tesseract struggles.

Next steps: Preprocessing, and improve config.

#### Creating a dataframe for the purpose of organising the above results

In [25]:
results_df = pd.DataFrame(results)

results_df

,file,type,text
0,docs\documents_ca2\1921_doc_sig_image.png,image,OPiS an ndnatoe Several Seorctanar |\nVeh Jacu...
1,docs\documents_ca2\aloevera_back.jpg,image,"See WET JUICE, weit; criabies US to capture tr..."
2,docs\documents_ca2\forms.jpg,image,"+ meet you we te yuan npr, anpostent ctl ao s ..."
3,docs\documents_ca2\drawing_using_words.bmp,image,Yt She\nSe A ge ee 9 tok gi aa\nvie? i Been wh...
4,docs\documents_ca2\1921_doc_1.pdf,pdf,"{1858. )Wt.5333-66.4000.12jl4.A, T.&Oo. ,Ltd. ..."
5,docs\documents_ca2\1921_doc_sign.pdf,pdf,DE/2/2(010)\n\nDE/2/2(020)\n\nDE/2/2(028)\n\nD...
6,docs\documents_ca2\cartoon_drawing.pdf,pdf,4\nLesson\nFamous Artists Cartoon Course\nThe ...
7,docs\documents_ca2\know_receipe.pdf,pdf,Soups Recipe Book \n1 \n\nContents \nRecipes \...
8,docs\documents_ca2\research_paper.pdf,pdf,\nFig. 1. Classification algorithms \nThe ide...
9,docs\documents_ca2\scikit_learn.pdf,pdf,"05/05/2026, 22:21 1.1. Linear Models — scikit-..."


The dataset contains a collection of mixed document types, including images and PDF files, all processed into a unified format for analysis. Each row represents a single document instance where the extracted text is obtained using OCR for images and a combination of direct text extraction and OCR fallback for PDFs. This allows both visual and textual content to be converted into machine-readable form. The resulting structure supports further NLP tasks such as text cleaning, feature extraction, and document understanding across heterogeneous sources within a single consistent dataset.

## Experiment 1 : Results:
represents the initial stage of the project, focusing on basic text extraction from images and PDF documents using OCR and document parsing techniques. This forms the foundation for later Vision preprocessing, NLP analysis, and multimodal integration.

(Including the page numbers for all the pdf. so, each page in the pdf is a row.)

In [28]:
image_files = jpg_files + bmp_files

results = []

for file in image_files:
    img = cv2.imread(file)

    processed = preprocess_image(img)
    custom_config = r'--oem 1 --psm 6'

    text = pytesseract.image_to_string(processed, config = custom_config)

    results.append({
        "file": file,
        "page": 0,
        "type": "image",
        "text": text
    })


#__________________________________

for file in pdf_files:
    doc = fitz.open(file)

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        # If text exists
        if text.strip():
            extracted_text = text

        # fallback OCR
        else:
            pix = page.get_pixmap(dpi=200)
            img = np.frombuffer(pix.samples, dtype=np.uint8)
            img = img.reshape(pix.height, pix.width, pix.n)

            custom_config = r'--oem 1 --psm 6'
            extracted_text = pytesseract.image_to_string(img, config=custom_config)

        results.append({
            "file": file,
            "page": page_num,
            "type": "pdf",
            "text": extracted_text
        })

In [29]:
results1_df = pd.DataFrame(results)

results1_df

,file,page,type,text
0,docs\documents_ca2\1921_doc_sig_image.png,0,image,OPiS an ndnatoe Several Seorctanar |\nVeh Jacu...
1,docs\documents_ca2\aloevera_back.jpg,0,image,"See WET JUICE, weit; criabies US to capture tr..."
2,docs\documents_ca2\forms.jpg,0,image,"+ meet you we te yuan npr, anpostent ctl ao s ..."
3,docs\documents_ca2\drawing_using_words.bmp,0,image,Yt She\nSe A ge ee 9 tok gi aa\nvie? i Been wh...
4,docs\documents_ca2\1921_doc_1.pdf,1,pdf,"{1858. )Wt.5333-66.4000.12jl4.A, T.&Oo. ,Ltd. ..."
5,docs\documents_ca2\1921_doc_1.pdf,2,pdf,A. W. W. Cotton left Amiens Street py \n... . ...
6,docs\documents_ca2\1921_doc_sign.pdf,1,pdf,DE/2/2(010)\n
7,docs\documents_ca2\1921_doc_sign.pdf,2,pdf,DE/2/2(020)\n
8,docs\documents_ca2\1921_doc_sign.pdf,3,pdf,DE/2/2(028)\n
9,docs\documents_ca2\1921_doc_sign.pdf,4,pdf,DE/2/2(048)\n


The current system is in the early exploratory stage, focusing on basic OCR-based extraction from images and PDFs. This forms the foundation for future Vision preprocessing, NLP analysis, and multimodal integration.

The dataset used in this task is a heterogeneous collection of documents designed to reflect real-world variability in document formats, layouts, and content complexity. It includes a mixture of scanned images, image-based text documents, and both digitally generated and scanned PDF files. The content spans a wide range of structures, including academic and technical text, recipe instructions, educational brochures, historical records, and diagram-heavy pages containing figures, tables, barcodes, stamps, seals, symbols, and signatures. Some documents follow a linear reading format, while others exhibit multi-column layouts, spatially arranged text, and embedded visual elements such as charts, maps, and illustrations.

The dataset contains both standalone images and multi-page PDFs, where each entry corresponds to either a full image or an individual page along with its extracted text, file type, and page number. A key characteristic is that several PDFs are image-based and do not contain embedded text, requiring Optical Character Recognition (OCR) for text extraction. While embedded text in digital PDFs has been successfully extracted, OCR performance varies across image types. For example, clearer images such as the aloevera document yield partial text extraction, whereas complex or stylised images like drawing_using_words and historical scanned documents (e.g., 1921 records) are more challenging and result in incomplete or noisy outputs. This highlights the importance of image preprocessing to improve OCR accuracy.

Overall, the dataset presents significant variability in structure and quality, making it suitable for real-world document understanding tasks and necessitating a hybrid approach that combines direct text extraction with OCR techniques.

## NEXT Step: NLP Processing:

Following text extraction, a series of Natural Language Processing (NLP) techniques are applied to clean, structure, and analyse the textual data. In the preprocessing stage, the extracted text is normalised by converting it to lowercase, removing noise such as special characters and punctuation using regular expressions, and eliminating common stop words that do not contribute meaningful information. The text is then tokenised into individual words or terms, and lemmatisation (or stemming) is applied to reduce words to their base form, improving consistency across the dataset.

Once the text is preprocessed, feature extraction techniques are used to transform the textual data into numerical representations suitable for analysis. Methods such as Term Frequency–Inverse Document Frequency (TF-IDF) or Bag-of-Words are applied to identify important words and patterns within the documents. In addition, more advanced techniques such as keyphrase extraction and Named Entity Recognition (NER) can be used to identify significant terms, entities (e.g., names, locations, organisations), and domain-specific keywords. Sentiment analysis may also be applied where relevant, particularly in descriptive or opinion-based text.

Finally, text analysis is performed to understand the structure and relationships within the documents. This includes categorising different sections such as titles, body text, and captions based on formatting and textual cues. Efforts can also be made to extract tabular data from structured layouts and to analyse relationships between text and visual elements, such as identifying references to figures, diagrams, or tables (e.g., “Fig. 1” or “Table 2”) within the text. These steps enable a deeper understanding of the document content beyond simple text extraction, supporting more advanced document interpretation tasks.

### STEP 1: PREPROCESS ALL DOCUMENTS

In [34]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    text = str(text).lower()

    # remove symbols and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # tokenization
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # lemmatization using spaCy
    doc = nlp(" ".join(tokens))

    lemmas = [token.lemma_ for token in doc]

    return " ".join(lemmas)

results1_df["clean_text"] = results1_df["text"].apply(preprocess_text)

results1_df

,file,page,type,text,clean_text
0,docs\documents_ca2\1921_doc_sig_image.png,0,image,OPiS an ndnatoe Several Seorctanar |\nVeh Jacu...,opis ndnatoe several seorctanar veh jacuary se...
1,docs\documents_ca2\aloevera_back.jpg,0,image,"See WET JUICE, weit; criabies US to capture tr...",see wet juice weit criabie we capture trie nat...
2,docs\documents_ca2\forms.jpg,0,image,"+ meet you we te yuan npr, anpostent ctl ao s ...",meet te yuan npr anpostent ctl ao meme yout so...
3,docs\documents_ca2\drawing_using_words.bmp,0,image,Yt She\nSe A ge ee 9 tok gi aa\nvie? i Been wh...,yt se ge ee tok gi aa vie whe oe dass bam bet ...
4,docs\documents_ca2\1921_doc_1.pdf,1,pdf,"{1858. )Wt.5333-66.4000.12jl4.A, T.&Oo. ,Ltd. ...",wt jl oo ltd wt j telegrams damp dublin teleph...
5,docs\documents_ca2\1921_doc_1.pdf,2,pdf,A. W. W. Cotton left Amiens Street py \n... . ...,w w cotton leave amiens street py p train belf...
6,docs\documents_ca2\1921_doc_sign.pdf,1,pdf,DE/2/2(010)\n,de
7,docs\documents_ca2\1921_doc_sign.pdf,2,pdf,DE/2/2(020)\n,de
8,docs\documents_ca2\1921_doc_sign.pdf,3,pdf,DE/2/2(028)\n,de
9,docs\documents_ca2\1921_doc_sign.pdf,4,pdf,DE/2/2(048)\n,de


In [35]:
# STEP 2 — TOKENS + POS TAGGING + LEMMAS

analysis_results = []

for index, row in results1_df.iterrows():

    doc = nlp(row["clean_text"])

    for token in doc:

        analysis_results.append({
            "file": row["file"],
            "page": row["page"],
            "token": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_,
            "explanation": spacy.explain(token.tag_)
        })

token_df = pd.DataFrame(analysis_results)

token_df.head(20)

,file,page,token,lemma,pos,tag,explanation
0,docs\documents_ca2\1921_doc_sig_image.png,0,opis,opis,PROPN,NNP,"noun, proper singular"
1,docs\documents_ca2\1921_doc_sig_image.png,0,ndnatoe,ndnatoe,VERB,VBG,"verb, gerund or present participle"
2,docs\documents_ca2\1921_doc_sig_image.png,0,several,several,ADJ,JJ,"adjective (English), other noun-modifier (Chin..."
3,docs\documents_ca2\1921_doc_sig_image.png,0,seorctanar,seorctanar,NOUN,NN,"noun, singular or mass"
4,docs\documents_ca2\1921_doc_sig_image.png,0,veh,veh,PROPN,NNP,"noun, proper singular"
5,docs\documents_ca2\1921_doc_sig_image.png,0,jacuary,jacuary,PROPN,NNP,"noun, proper singular"
6,docs\documents_ca2\1921_doc_sig_image.png,0,senretary,senretary,PROPN,NNP,"noun, proper singular"
7,docs\documents_ca2\1921_doc_sig_image.png,0,piranate,piranate,ADJ,JJR,"adjective, comparative"
8,docs\documents_ca2\1921_doc_sig_image.png,0,vv,vv,ADP,IN,"conjunction, subordinating or preposition"
9,docs\documents_ca2\1921_doc_sig_image.png,0,tiere,tiere,VERB,VBN,"verb, past participle"


In [ ]:
page_results = []

for _, row in results1_df.iterrows():

    doc = nlp(row["clean_text"])

    token_list = []

    for token in doc:
        token_list.append({
            "token": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_
        })

    page_results.append({
        "file": row["file"],
        "page": row["page"],
        "tokens": token_list
    })

page_df = pd.DataFrame(page_results)

In [36]:
# STEP 3 — NAMED ENTITY RECOGNITION (NER)

ner_results = []

for index, row in results1_df.iterrows():

    doc = nlp(row["clean_text"])

    for ent in doc.ents:

        ner_results.append({
            "file": row["file"],
            "page": row["page"],
            "entity": ent.text,
            "label": ent.label_,
            "meaning": spacy.explain(ent.label_)
        })

ner_df = pd.DataFrame(ner_results)

ner_df.head(20)

,file,page,entity,label,meaning
0,docs\documents_ca2\1921_doc_sig_image.png,0,laceh,GPE,"Countries, cities, states"
1,docs\documents_ca2\1921_doc_sig_image.png,0,teara fate tnep tre,FAC,"Buildings, airports, highways, bridges, etc."
2,docs\documents_ca2\1921_doc_sig_image.png,0,aireastte de,ORG,"Companies, agencies, institutions, etc."
3,docs\documents_ca2\aloevera_back.jpg,0,mio,NORP,Nationalities or religious or political groups
4,docs\documents_ca2\aloevera_back.jpg,0,conte tuttliz,PERSON,"People, including fictional"
5,docs\documents_ca2\aloevera_back.jpg,0,al riparo dalla,PERSON,"People, including fictional"
6,docs\documents_ca2\aloevera_back.jpg,0,aloe vera,PERSON,"People, including fictional"
7,docs\documents_ca2\aloevera_back.jpg,0,cee tog aloe,PERSON,"People, including fictional"
8,docs\documents_ca2\aloevera_back.jpg,0,kay aes aloe,PERSON,"People, including fictional"
9,docs\documents_ca2\forms.jpg,0,det ndecate,ORG,"Companies, agencies, institutions, etc."


In [37]:
# STEP 4 — TF-IDF FEATURE EXTRACTION

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=100)

X = tfidf.fit_transform(results1_df["clean_text"])

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# STEP 5 — BAG OF WORDS

from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=100)

bow_matrix = bow.fit_transform(results1_df["clean_text"])

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow.get_feature_names_out()
)

bow_df.head()

In [ ]:
# STEP 6 — KEY PHRASE EXTRACTION

keyphrases = []

for index, row in results1_df.iterrows():

    doc = nlp(row["clean_text"])

    phrases = [chunk.text for chunk in doc.noun_chunks]

    keyphrases.append({
        "file": row["file"],
        "page": row["page"],
        "key_phrases": phrases
    })

keyphrase_df = pd.DataFrame(keyphrases)

keyphrase_df.head()

In [ ]:
# STEP 7 — TEXT ANALYSIS

results1_df["has_figure_reference"] = results1_df["text"].str.contains(
    r'Fig|Figure|Table|Diagram',
    case=False,
    regex=True
)

results1_df[
    ["file", "page", "has_figure_reference"]
].head(20)

In [ ]:
# STEP 8 — SIMPLE SECTION CATEGORISATION

def classify_section(text):

    text = str(text)

    if len(text.split()) < 15:
        return "Title/Caption"

    elif "figure" in text.lower() or "fig." in text.lower():
        return "Figure Reference"

    else:
        return "Body Text"

results1_df["section_type"] = results1_df["text"].apply(classify_section)

results1_df[
    ["file", "page", "section_type"]
].head(20)

In [ ]:
stopwords = nlp.Defaults.stop_words

In [ ]:
# Get word index
word_index = tf.keras.datasets.imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

# Clean decode + remove stopwords
def clean_decode_review(encoded_review):
    text = " ".join([reverse_word_index.get(i - 3, "") for i in encoded_review if i >= 3])
    text = text.lower()
    text = re.sub(r'\b(br|[0-9]+)\b', '', text)  # remove br and numbers
    text = re.sub(r'[^\w\s]', '', text)          # remove punctuation
    words = [w for w in text.split() if w and w != "?" and w not in stopwords]
    return " ".join(words)


# Apply to first 1000 reviews
num_reviews = 1000
texts = [clean_decode_review(review) for review in train_data[:num_reviews]]

# Build DataFrame
data = pd.DataFrame({
    "text": texts,
    "sentiment": train_labels[:num_reviews]
})
data['sentiment'] = data['sentiment'].map({1: 'positive', 0: 'negative'})

data.head()

In [ ]:


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)   # remove symbols
    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

results1_df["clean_text"] = results1_df["text"].apply(preprocess)

In [ ]:
# Feature Extraction

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(results1_df["clean_text"])

In [ ]:
# Bag of words

from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=1000)
X_bow = bow.fit_transform(results1_df["clean_text"])

TF-IDF helps identify the most informative words in each document, which can be used to understand the main topic or theme of that document.

Recipe document

TF-IDF keywords:

soup
ingredients
cook
tomato

👉 You conclude:
✔ “This is a recipe document”

🔵 Scikit-learn document

TF-IDF keywords:

regression
model
coefficient
data

👉 You conclude:
✔ “This is a technical/machine learning document”

🟡 Trinity brochure

TF-IDF keywords:

student
campus
degree
university

👉 You conclude:
✔ “This is an educational brochure”

In [ ]:
# Named entity recognition


def get_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

results1_df["entities"] = results1_df["text"].apply(get_entities)

In [ ]:
# Text Analysis

def classify_text(text):
    if len(text) < 50:
        return "Title/Caption"
    elif "fig" in text.lower():
        return "Figure Reference"
    else:
        return "Body"

results1_df["section"] = results1_df["text"].apply(classify_text)

In [ ]:
# Detect tables

def detect_table(text):
    if "\t" in text or "  " in text:
        return "Table-like"
    return "No Table"

results1_df["table_detected"] = results1_df["text"].apply(detect_table)

In [ ]:
# Keu phrases

def keywords(text):
    return text.split()[:10]

results1_df["keywords"] = results1_df["clean_text"].apply(keywords)

The system follows a pipeline approach starting with OCR-based text extraction from images and scanned PDFs, combined with direct text extraction for digital PDFs. The extracted text is then preprocessed using tokenisation, stopword removal, and lemmatisation. Feature extraction is performed using TF-IDF and Bag-of-Words models to represent textual information numerically. Further analysis includes named entity recognition, keyword extraction, and heuristic-based classification of document sections such as titles, body text, and captions. Additional processing identifies tabular patterns and references to figures or diagrams. The final output is a structured dataset containing both textual content and extracted semantic insights.

# Image Processing & Feature Detection

In [ ]:
# Image Preprocessing

import cv2

def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # noise removal
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    # thresholding (important for OCR)
    thresh = cv2.threshold(blur, 0, 255,
                           cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

    return thresh

In [ ]:
# Feature deetection

edges = cv2.Canny(img, 50, 150)

In [ ]:
# Contour Detection

contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

img_copy = img.copy()
cv2.drawContours(img_copy, contours, -1, (0,255,0), 2)

In [ ]:
# ORB

orb = cv2.ORB_create()
keypoints, descriptors = orb.detectAndCompute(gray, None)

img_kp = cv2.drawKeypoints(img, keypoints, None)

In [ ]:
# Imnage Segmentation

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)

segmented = img.copy()

for i in range(1, num_labels):
    x, y, w, h, area = stats[i]

    if area > 500:  # filter noise
        cv2.rectangle(segmented, (x,y), (x+w, y+h), (255,0,0), 2)

In [ ]:
# Image enhancement

# contrast improvement
alpha = 1.5   # contrast control
beta = 0      # brightness

enhanced = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

# PART 3 Multi-modal Integration

In [ ]:
# STEP 1 — Combine text + image results


results1_df

In [ ]:
results1_df["num_regions"] = None

for i, row in results1_df.iterrows():
    if row["type"] == "image":
        img = cv2.imread(row["file"])
        processed = preprocess_image(img)

        num_labels, _, _, _ = cv2.connectedComponentsWithStats(processed)
        results1_df.at[i, "num_regions"] = num_labels

In [ ]:
# Showig visual outputs

import matplotlib.pyplot as plt

plt.imshow(segmented)
plt.title("Segmented Image")
plt.axis("off")

### STEP 3 — Final Output (what to submit)

You should present:

✔ 1. Text Results
cleaned text
TF-IDF / keywords
entities
✔ 2. Image Results
original image
processed image
detected contours
segmented regions
✔ 3. Insights

Examples you can write:

Documents contain multi-column layouts
Tables detected via structured regions
Figures identified using contour density
OCR accuracy improves after preprocessing
Some documents (e.g. scanned images) require heavy enhancement
✔ How to explain this in your assignment

Write this:

Image Processing and Feature Detection (paragraph)

Image preprocessing was performed using OpenCV to enhance document quality prior to analysis. Techniques such as grayscale conversion, Gaussian blurring, and adaptive thresholding were applied to reduce noise and improve text visibility. Feature detection was carried out using edge detection (Canny) and contour detection to identify structural elements such as text blocks, tables, and diagrams. Additionally, ORB feature detection was used to identify key points in visually complex regions. Image segmentation was achieved using connected component analysis to isolate meaningful regions within the document, such as separating text areas from graphical content. Image enhancement techniques, including contrast adjustment and binarisation, further improved the clarity and legibility of the extracted content.

Multi-modal Integration (paragraph)

The system integrates both text and image processing outputs to provide a unified understanding of the documents. Extracted text data, along with preprocessing and feature extraction results, are combined with visual features obtained from image analysis. The final output includes structured text, detected entities, segmented image regions, and identified visual patterns such as tables and diagrams. This integrated approach enables better interpretation of complex documents by linking textual information with corresponding visual elements.

The selection of image preprocessing techniques was guided by the characteristics of each document type. For scanned and low-quality images, noise reduction and thresholding were applied to improve OCR performance. For documents containing structural elements such as tables and diagrams, edge detection and morphological operations were used to enhance feature detection. Image segmentation techniques such as connected component analysis were employed to isolate meaningful regions within the document. The effectiveness of preprocessing was evaluated by comparing OCR outputs before and after processing, assessing improvements in text clarity, readability, and reduction of noise. Visual inspection and feature detection results, such as clearer edges and more accurate region segmentation, were also used to determine the quality of preprocessing.


For text-based images, preprocessing techniques were evaluated by comparing OCR outputs before and after processing, selecting methods that improved text clarity and readability. For non-text images such as diagrams and drawings, evaluation was based on visual quality and effectiveness of feature detection techniques such as edge detection and contour analysis. Clearer edges, better segmentation, and meaningful feature detection indicated successful preprocessing.

✔ Final understanding (very important)

You are doing adaptive processing:

If image has text → optimise for OCR
If image has visuals → optimise for feature detection

In [ ]:
# step 6: NLP

texts = [item["text"] for item in results]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)

# Vision

## Sub Heading 1

In [ ]:
# code here...

# Multi-modal

## Sub Heading 1

In [ ]:
# code here

# Final Output

In [ ]:
# code

In [ ]:
# Re

## References
1. https://nationalarchives.ie/